# 🏏 Cricbuzz LiveStats: Data Fetching & ETL Testing Notebook
This interactive notebook demonstrates:
1. Validating database connectivity and schema
2. Prototyping Cricbuzz REST API endpoints (Live matches, scorecards, commentary)
3. Running ETL ingestion: syncing completed matches into PostgreSQL
4. Inspecting database tables and player leaderboards

In [ ]:
# Step 1: Import modules and check database status
import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

from utils.db_connection import check_db_status, execute_query
from utils.api_handler import (
    get_live_matches, 
    get_match_details, 
    get_match_commentary, 
    get_top_player_leaderboards,
    sync_completed_match_to_db
)

status = check_db_status()
print('Database Status:', status['status'])
print('Dialect:', status['dialect'])
print('Table Count:', status['table_count'])
print('Tables:', sorted(status['tables']))

### Step 2: Fetch Live Matches via API

In [ ]:
matches = get_live_matches()
print(f'Total live matches retrieved: {len(matches)}')
for m in matches:
    print(f"• {m['description']} [{m['format']}] - {m['status']}")

### Step 3: Inspect Live Scorecard & Commentary

In [ ]:
if matches:
    match_id = matches[0]['id']
    details = get_match_details(match_id)
    print(f"Match: {details['description']}")
    print(f"Venue: {details['venue']}")
    print(f"Current Status: {details['status']}")
    
    state = details.get('current_state', {})
    if state:
        print(f"Batting Team: {state.get('batting_team')} | CRR: {state.get('crr')}")
        print('Active Batters:')
        for b in state.get('batters', []):
            strike = '*' if b.get('on_strike') else ''
            print(f"  - {b['name']}{strike}: {b['runs']} ({b['balls']}b, SR: {b['sr']})")
            
    print('\nRecent Commentary:')
    commentary = get_match_commentary(match_id)
    for c in commentary[:3]:
        print(f"  Over {c['over']}: {c['text']}")

### Step 4: Sync Match to PostgreSQL Database

In [ ]:
if matches:
    sync_res = sync_completed_match_to_db(matches[0]['id'])
    print('Sync Response:', sync_res)

### Step 5: Query Leading Player Stats from PostgreSQL

In [ ]:
df_top_odi = execute_query('''
    SELECT p.full_name, p.country, cs.total_runs, cs.batting_avg, cs.centuries, cs.highest_score
    FROM player_career_stats cs
    JOIN players p ON cs.player_id = p.player_id
    WHERE cs.format = 'ODI'
    ORDER BY cs.total_runs DESC
    LIMIT 10;
')
df_top_odi